In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [30]:
dataset_list = ["ARC_Challenge", "CommonSenseQA", "MMLU", "OpenBookQA"
                ]
model_list = [
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B",
    "Qwen/Qwen1.5-4B",
    "Qwen/Qwen1.5-0.5B",
    "Qwen/Qwen1.5-1.8B",
    "openai-community/gpt2",
    "openai-community/gpt2-large",
    "openai-community/gpt2-medium",
  ]

In [31]:
def get_mean_std(data): 
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)
    return mean, std

In [32]:
def get_data_list(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        results = [json.loads(line) for line in f]

    group_base = ["prompt_1", "prompt_2", "prompt_3"]
    group_modify = ["prompt_4", "prompt_5", "prompt_6", "prompt_7", "prompt_8", "prompt_9"]
    group_misalign = ["prompt_10", "prompt_11", "prompt_12", "prompt_13", "prompt_14", "prompt_15"]
    prompt_1_list = np.stack([r["prompt_key0"] for r in results], axis=0)  
    prompt_2_list = np.stack([r["prompt_key1"] for r in results], axis=0) 

    all_grad_list = np.stack([r["grads_norm_list"] for r in results], axis=0)              # shape: (num_samples, dim)
    all_delta_z_list = np.stack([r["delta_z_norm_list"] for r in results], axis=0)
    
    delta_log_prob_norm_list = [r["delta_log_prob_norm"] for r in results]

    modify_grad_list = []
    modify_delta_z_list = []
    modify_delta_log_prob_norm_list = []
    misalign_grad_list = []
    misalign_delta_z_list = []
    misalign_delta_log_prob_norm_list = []

    for p1, p2, grad, delta_z, delta_log_prob in zip(prompt_1_list, prompt_2_list, all_grad_list, all_delta_z_list, delta_log_prob_norm_list):
        if p1 == p2:
            continue
        if p1 in group_base and p2 in group_modify:
            modify_grad_list.append(grad)
            modify_delta_z_list.append(delta_z)
            modify_delta_log_prob_norm_list.append(delta_log_prob)
        elif p1 in group_base and p2 in group_misalign:
            misalign_grad_list.append(grad)
            misalign_delta_z_list.append(delta_z)
            misalign_delta_log_prob_norm_list.append(delta_log_prob)

    modify_grad_list = np.array(modify_grad_list)
    modify_delta_z_list = np.array(modify_delta_z_list)
    misalign_grad_list = np.array(misalign_grad_list)
    misalign_delta_z_list = np.array(misalign_delta_z_list)
    modify_upper_bound_list = modify_grad_list * modify_delta_z_list
    misalign_upper_bound_list = misalign_grad_list * misalign_delta_z_list

    modify_grad_mean_list, modify_grad_std_list = get_mean_std(modify_grad_list)
    modify_delta_z_mean_list, modify_delta_z_std_list = get_mean_std(modify_delta_z_list)
    misalign_grad_mean_list, misalign_grad_std_list = get_mean_std(misalign_grad_list)
    misalign_delta_z_mean_list, misalign_delta_z_std_list = get_mean_std(misalign_delta_z_list)
    modify_upper_bound_mean_list, modify_upper_bound_std_list = get_mean_std(modify_upper_bound_list)
    misalign_upper_bound_mean_list, misalign_upper_bound_std_list = get_mean_std(misalign_upper_bound_list)

    modify_result_dict = {
        "grad_mean_list": modify_grad_mean_list,
        "grad_std_list": modify_grad_std_list,
        "delta_z_mean_list": modify_delta_z_mean_list,
        "delta_z_std_list": modify_delta_z_std_list,
        "upper_bound_mean_list": modify_upper_bound_mean_list,
        "upper_bound_std_list": modify_upper_bound_std_list,
        "delta_log_prob_norm_list": modify_delta_log_prob_norm_list,
    }

    misalign_result_dict = {
        "grad_mean_list": misalign_grad_mean_list,
        "grad_std_list": misalign_grad_std_list,
        "delta_z_mean_list": misalign_delta_z_mean_list,
        "delta_z_std_list": misalign_delta_z_std_list,
        "upper_bound_mean_list": misalign_upper_bound_mean_list,
        "upper_bound_std_list": misalign_upper_bound_std_list,
        "delta_log_prob_norm_list": misalign_delta_log_prob_norm_list,
    }
    return modify_result_dict, misalign_result_dict

In [33]:
def plot_line(modify_delta_z_mean_list, 
              modify_delta_z_std_list,
              misalign_delta_z_mean_list, 
              misalign_delta_z_std_list,
              modify_color, 
              misalign_color, 
              dataset, 
              model_name_or_path, 
              label):
    plt.figure(figsize=(2.5, 2.5))

    x = np.arange(len(modify_delta_z_mean_list))
    xticks = [str(i) for i in range(len(x))]
    step = max(1, len(x) // 4)
    
    modify_mean = np.array(modify_delta_z_mean_list)
    modify_std = np.array(modify_delta_z_std_list)
    modify_lower = modify_mean - modify_std
    modify_upper = modify_mean + modify_std

    misalign_mean = np.array(misalign_delta_z_mean_list)
    misalign_std = np.array(misalign_delta_z_std_list)
    misalign_lower = misalign_mean - misalign_std
    misalign_upper = misalign_mean + misalign_std

    plt.plot(
        x,
        modify_delta_z_mean_list,
        label=r"Modify",
        color=modify_color,
        linewidth=2
    )
    plt.plot(
        x,
        misalign_delta_z_mean_list,
        label=r"Misalign",
        color=misalign_color,
        linewidth=2
    )
    plt.fill_between(
        x,
        modify_lower,
        modify_upper,
        color=modify_color,
        alpha=0.2,
        linewidth=0
    )
    plt.fill_between(
        x,
        misalign_lower,
        misalign_upper,
        color=misalign_color,
        alpha=0.2,
        linewidth=0
    )
    # if line is not None:
    #     plt.axhline(y=line, color="#0095FF", linestyle="-", linewidth=2)

    plt.xticks(x[::step], xticks[::step])

    plt.xlabel("Number of layers")
    plt.title("Modify vs. Misalign")
    plt.legend()
    plt.grid(False)
    plt.tight_layout()
    
    save_path = f"../../results/figure_results/how_modify_misalign/{model_name_or_path}/{dataset}_{label}.pdf"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path)
    plt.close()


In [34]:
def plot_grads_and_deltaz(dataset, model_name_or_path):
    file_path = f"../../results/data_results/misalignment/{model_name_or_path}/{dataset}_result.jsonl"
    modify_result_dict, misalign_result_dict = get_data_list(file_path)
    
    modify_grad_mean_list = modify_result_dict["grad_mean_list"]
    modify_grad_std_list = modify_result_dict["grad_std_list"]
    modify_delta_z_mean_list = modify_result_dict["delta_z_mean_list"]
    modify_delta_z_std_list = modify_result_dict["delta_z_std_list"]
    modify_upper_bound_mean_list = modify_result_dict["upper_bound_mean_list"]
    modify_upper_bound_std_list = modify_result_dict["upper_bound_std_list"]
    modify_delta_log_prob_norm_list = modify_result_dict["delta_log_prob_norm_list"]

    misalign_grad_mean_list = misalign_result_dict["grad_mean_list"]
    misalign_grad_std_list = misalign_result_dict["grad_std_list"]
    misalign_delta_z_mean_list = misalign_result_dict["delta_z_mean_list"]
    misalign_delta_z_std_list = misalign_result_dict["delta_z_std_list"]
    misalign_upper_bound_mean_list = misalign_result_dict["upper_bound_mean_list"]
    misalign_upper_bound_std_list = misalign_result_dict["upper_bound_std_list"]
    misalign_delta_log_prob_norm_list = misalign_result_dict["delta_log_prob_norm_list"]

    modify_mean_delta_log_prob_norm = np.mean(modify_delta_log_prob_norm_list)
    misalign_mean_delta_log_prob_norm = np.mean(misalign_delta_log_prob_norm_list)

    modify_color = "#1E89BF"
    misalign_color = "#F74316"
    plot_line(modify_delta_z_mean_list, 
              modify_delta_z_std_list,
              misalign_delta_z_mean_list, 
              misalign_delta_z_std_list,
              modify_color, 
              misalign_color, 
              dataset, 
              model_name_or_path, 
              "modify_misalign")
   

In [35]:
for dataset in dataset_list:
    for model_name_or_path in model_list:
        plot_grads_and_deltaz(dataset, model_name_or_path)
